In [1]:
import os

In [2]:
file_path=r"C:\Users\HP PC\Downloads\train"
os.listdir(file_path)

['images', 'labels', 'labels.cache', 'NORMAL', 'PNEUMONIA']

In [3]:
file_path=r"C:\Users\HP PC\Downloads\test"
os.listdir(file_path)

['images', 'labels', 'NORMAL', 'PNEUMONIA']

### STEP-BY-STEP CLEAN DATASET CREATION

### Imports

In [4]:
import os
import shutil

### Original dataset paths

In [5]:
train_original = r"C:\Users\HP PC\Downloads\train"
test_original  = r"C:\Users\HP PC\Downloads\test"

### New clean dataset paths

In [6]:
clean_train = r"C:\Users\HP PC\Downloads\clean_train"
clean_test  = r"C:\Users\HP PC\Downloads\clean_test"

### Create clean folders

In [7]:
os.makedirs(clean_train, exist_ok=True)
os.makedirs(clean_test, exist_ok=True)

### Copy ONLY valid classes for TRAIN

In [8]:
for cls in ["NORMAL", "PNEUMONIA"]:
    
    src = os.path.join(train_original, cls)
    dst = os.path.join(clean_train, cls)
    
    if os.path.exists(dst):
        shutil.rmtree(dst)
    
    shutil.copytree(src, dst)

print("Clean train dataset created")

Clean train dataset created


### Copy ONLY valid classes for TEST

In [9]:
for cls in ["NORMAL", "PNEUMONIA"]:
    
    src = os.path.join(test_original, cls)
    dst = os.path.join(clean_test, cls)
    
    if os.path.exists(dst):
        shutil.rmtree(dst)
    
    shutil.copytree(src, dst)

print("Clean test dataset created")

Clean test dataset created


### Verify clean structure

In [10]:
print(os.listdir(clean_train))

['NORMAL', 'PNEUMONIA']


In [11]:
print(os.listdir(clean_test))

['NORMAL', 'PNEUMONIA']


### Create validation set

Since original dataset has no validation folder.

### Validation path

In [12]:
clean_val = r"C:\Users\HP PC\Downloads\clean_val"

os.makedirs(clean_val, exist_ok=True)

### Split 20% from train

In [13]:
import random

for cls in ["NORMAL", "PNEUMONIA"]:
    
    os.makedirs(os.path.join(clean_val, cls), exist_ok=True)

    files = os.listdir(os.path.join(clean_train, cls))
    
    random.shuffle(files)

    split_size = int(0.2 * len(files))

    val_files = files[:split_size]

    for f in val_files:
        
        src = os.path.join(clean_train, cls, f)
        dst = os.path.join(clean_val, cls, f)

        shutil.move(src, dst)

print("Validation dataset created")

Validation dataset created


### Final verification

In [14]:
print(os.listdir(clean_train))
print(os.listdir(clean_val))
print(os.listdir(clean_test))

['NORMAL', 'PNEUMONIA']
['NORMAL', 'PNEUMONIA']
['NORMAL', 'PNEUMONIA']


### Verify image counts

This is VERY important.

In [15]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.densenet import preprocess_input

datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_generator = datagen.flow_from_directory(
    clean_train,
    target_size=(224,224),
    batch_size=32,
    class_mode='binary'
)

val_generator = datagen.flow_from_directory(
    clean_val,
    target_size=(224,224),
    batch_size=32,
    class_mode='binary'
)

test_generator = datagen.flow_from_directory(
    clean_test,
    target_size=(224,224),
    batch_size=32,
    class_mode='binary'
)

Found 4187 images belonging to 2 classes.
Found 1045 images belonging to 2 classes.
Found 624 images belonging to 2 classes.


### Train DenseNet121

In [16]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator

### Data generators
🔹 Train generator

In [17]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=15,
    zoom_range=0.2,
    horizontal_flip=True
)

### Validation/Test generator

In [18]:
test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

### Load datasets

In [19]:
train_generator = train_datagen.flow_from_directory(
    clean_train,
    target_size=(224,224),
    batch_size=32,
    class_mode='binary'
)

val_generator = test_datagen.flow_from_directory(
    clean_val,
    target_size=(224,224),
    batch_size=32,
    class_mode='binary',
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    clean_test,
    target_size=(224,224),
    batch_size=32,
    class_mode='binary',
    shuffle=False
)

Found 4187 images belonging to 2 classes.
Found 1045 images belonging to 2 classes.
Found 624 images belonging to 2 classes.


### Build DenseNet121

In [20]:
base_model = DenseNet121(
    include_top=False,
    weights='imagenet',
    input_shape=(224,224,3)
)

29084464/29084464 [==============================] - 105s 4us/step


### Freeze base model initially

In [21]:
base_model.trainable = False

### Classification head

In [22]:
x = layers.GlobalAveragePooling2D()(base_model.output)

x = layers.BatchNormalization()(x)

x = layers.Dense(128, activation='relu')(x)

x = layers.Dropout(0.4)(x)

output = layers.Dense(1, activation='sigmoid')(x)

model = models.Model(
    inputs=base_model.input,
    outputs=output
)

### Compile

In [24]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

### Initial training

In [25]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10
)

Epoch 1/10
131/131 [==============================] - 362s 3s/step - loss: 0.1821 - accuracy: 0.9229 - val_loss: 0.1065 - val_accuracy: 0.9713
Epoch 2/10
131/131 [==============================] - 337s 3s/step - loss: 0.1239 - accuracy: 0.9477 - val_loss: 0.1106 - val_accuracy: 0.9531
Epoch 3/10
131/131 [==============================] - 337s 3s/step - loss: 0.1013 - accuracy: 0.9620 - val_loss: 0.1038 - val_accuracy: 0.9560
Epoch 4/10
131/131 [==============================] - 333s 3s/step - loss: 0.0994 - accuracy: 0.9651 - val_loss: 0.0989 - val_accuracy: 0.9579
Epoch 5/10
131/131 [==============================] - 332s 3s/step - loss: 0.0958 - accuracy: 0.9639 - val_loss: 0.1350 - val_accuracy: 0.9426
Epoch 6/10
131/131 [==============================] - 340s 3s/step - loss: 0.0785 - accuracy: 0.9709 - val_loss: 0.1479 - val_accuracy: 0.9378
Epoch 7/10
131/131 [==============================] - 332s 3s/step - loss: 0.0849 - accuracy: 0.9670 - val_loss: 0.1485 - val_accuracy: 0.9397

### Evaluate DenseNet now

In [26]:
test_loss, test_acc = model.evaluate(test_generator)

print("Test Accuracy:", test_acc)

20/20 [==============================] - 34s 2s/step - loss: 0.3347 - accuracy: 0.8942
Test Accuracy: 0.8942307829856873


### Then confusion matrix

In [27]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

y_probs = model.predict(test_generator)

y_pred = (y_probs > 0.5).astype(int)

print(confusion_matrix(test_generator.classes, y_pred))
print(classification_report(test_generator.classes, y_pred))

20/20 [==============================] - 35s 2s/step
[[187  47]
 [ 19 371]]
              precision    recall  f1-score   support

           0       0.91      0.80      0.85       234
           1       0.89      0.95      0.92       390

    accuracy                           0.89       624
   macro avg       0.90      0.88      0.88       624
weighted avg       0.90      0.89      0.89       624



### COMPLETE STEP-BY-STEP MOBILENETV2 PIPELINE

### Imports

In [28]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator

### Dataset paths

Use your clean folders.

In [29]:
train_path = r"C:\Users\HP PC\Downloads\clean_train"
val_path   = r"C:\Users\HP PC\Downloads\clean_val"
test_path  = r"C:\Users\HP PC\Downloads\clean_test"

### Data generators

#### Training generator

In [30]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=15,
    zoom_range=0.2,
    horizontal_flip=True
)

#### Validation/Test generator

In [31]:
test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

#### Load datasets

In [32]:
train_generator = train_datagen.flow_from_directory(
    train_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='binary'
)

val_generator = test_datagen.flow_from_directory(
    val_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='binary',
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    test_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='binary',
    shuffle=False
)

Found 4187 images belonging to 2 classes.
Found 1045 images belonging to 2 classes.
Found 624 images belonging to 2 classes.


### Build MobileNetV2

In [33]:
base_model = MobileNetV2(
    include_top=False,
    weights='imagenet',
    input_shape=(224,224,3)
)

#### Freeze base model initially

In [34]:
base_model.trainable = False

### Build classification head

This setup worked well for you previously.

In [35]:
x = layers.GlobalAveragePooling2D()(base_model.output)

x = layers.BatchNormalization()(x)

x = layers.Dense(128, activation='relu')(x)

x = layers.Dropout(0.4)(x)

output = layers.Dense(1, activation='sigmoid')(x)

model = models.Model(
    inputs=base_model.input,
    outputs=output
)

### Compile model

In [36]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

### Add callbacks (IMPORTANT)

This improves stability.

### EarlyStopping

In [38]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

### Reduce LR automatically

In [39]:
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=1,
    min_lr=1e-6
)

### Initial training

In [40]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/10
131/131 [==============================] - 147s 1s/step - loss: 0.2212 - accuracy: 0.9128 - val_loss: 0.1086 - val_accuracy: 0.9646 - lr: 0.0010
Epoch 2/10
131/131 [==============================] - 148s 1s/step - loss: 0.1431 - accuracy: 0.9424 - val_loss: 0.0850 - val_accuracy: 0.9684 - lr: 0.0010
Epoch 3/10
131/131 [==============================] - 148s 1s/step - loss: 0.1168 - accuracy: 0.9594 - val_loss: 0.0849 - val_accuracy: 0.9646 - lr: 0.0010
Epoch 4/10
131/131 [==============================] - 149s 1s/step - loss: 0.1096 - accuracy: 0.9606 - val_loss: 0.0798 - val_accuracy: 0.9617 - lr: 0.0010
Epoch 5/10
131/131 [==============================] - 151s 1s/step - loss: 0.1057 - accuracy: 0.9587 - val_loss: 0.0785 - val_accuracy: 0.9703 - lr: 0.0010
Epoch 6/10
131/131 [==============================] - 152s 1s/step - loss: 0.0964 - accuracy: 0.9627 - val_loss: 0.0955 - val_accuracy: 0.9608 - lr: 0.0010
Epoch 7/10
131/131 [==============================] - 160s 1s/st

In [41]:
test_loss, test_acc = model.evaluate(test_generator)

print("Test Accuracy:", test_acc)

20/20 [==============================] - 10s 463ms/step - loss: 0.4597 - accuracy: 0.8606
Test Accuracy: 0.8605769276618958


In [42]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

y_probs = model.predict(test_generator)

y_pred = (y_probs > 0.5).astype(int)

print(confusion_matrix(test_generator.classes, y_pred))

print(classification_report(
    test_generator.classes,
    y_pred
))

20/20 [==============================] - 11s 466ms/step
[[153  81]
 [  6 384]]
              precision    recall  f1-score   support

           0       0.96      0.65      0.78       234
           1       0.83      0.98      0.90       390

    accuracy                           0.86       624
   macro avg       0.89      0.82      0.84       624
weighted avg       0.88      0.86      0.85       624



### STEP-BY-STEP STABLE MOBILENETV2 FINE-TUNING

### Unfreeze MobileNet carefully

In [43]:
base_model.trainable = True

### Freeze MOST layers

Only last layers train.

In [44]:
for layer in base_model.layers[:-20]:
    layer.trainable = False

### Freeze BatchNorm layers (VERY IMPORTANT)

This stabilizes MobileNet greatly.

In [45]:
import tensorflow as tf

for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

### Recompile with VERY LOW LR

In [46]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

### Add callbacks

### EarlyStopping

In [47]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

### Reduce LR

In [48]:
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=1,
    min_lr=1e-7
)

#### Fine-tune gently

In [49]:
history_fine = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=6,
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/6
131/131 [==============================] - 199s 1s/step - loss: 0.0887 - accuracy: 0.9637 - val_loss: 0.0918 - val_accuracy: 0.9608 - lr: 1.0000e-05
Epoch 2/6
131/131 [==============================] - 200s 2s/step - loss: 0.0922 - accuracy: 0.9661 - val_loss: 0.0846 - val_accuracy: 0.9665 - lr: 1.0000e-05
Epoch 3/6
131/131 [==============================] - 204s 2s/step - loss: 0.0814 - accuracy: 0.9687 - val_loss: 0.1353 - val_accuracy: 0.9483 - lr: 1.0000e-05
Epoch 4/6
131/131 [==============================] - 196s 1s/step - loss: 0.0768 - accuracy: 0.9699 - val_loss: 0.0882 - val_accuracy: 0.9656 - lr: 5.0000e-06


In [50]:
test_loss, test_acc = model.evaluate(test_generator)

print("Test Accuracy:", test_acc)

20/20 [==============================] - 15s 694ms/step - loss: 0.4350 - accuracy: 0.8670
Test Accuracy: 0.8669871687889099


In [51]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

y_probs = model.predict(test_generator)

y_pred = (y_probs > 0.5).astype(int)

print(confusion_matrix(test_generator.classes, y_pred))

print(classification_report(
    test_generator.classes,
    y_pred
))

20/20 [==============================] - 15s 689ms/step
[[156  78]
 [  5 385]]
              precision    recall  f1-score   support

           0       0.97      0.67      0.79       234
           1       0.83      0.99      0.90       390

    accuracy                           0.87       624
   macro avg       0.90      0.83      0.85       624
weighted avg       0.88      0.87      0.86       624



### Save model here

In [52]:
model.save_weights("mobilenetv2_pneumonia_weights.h5")

print("MobileNetV2 weights saved successfully")

MobileNetV2 weights saved successfully


### Let us try threhold tunning

### Get prediction probabilities

In [53]:
y_probs = model.predict(test_generator)

20/20 [==============================] - 15s 736ms/step


### Test threshold = 0.50 (baseline)

In [54]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

threshold = 0.50

y_pred = (y_probs > threshold).astype(int)

print("Threshold:", threshold)

print(confusion_matrix(test_generator.classes, y_pred))

print(classification_report(
    test_generator.classes,
    y_pred
))

Threshold: 0.5
[[156  78]
 [  5 385]]
              precision    recall  f1-score   support

           0       0.97      0.67      0.79       234
           1       0.83      0.99      0.90       390

    accuracy                           0.87       624
   macro avg       0.90      0.83      0.85       624
weighted avg       0.88      0.87      0.86       624



### Test threshold = 0.55

In [57]:
threshold = 0.55

y_pred = (y_probs > threshold).astype(int)

print("Threshold:", threshold)

print(confusion_matrix(test_generator.classes, y_pred))

print(classification_report(
    test_generator.classes,
    y_pred
))

Threshold: 0.55
[[162  72]
 [  5 385]]
              precision    recall  f1-score   support

           0       0.97      0.69      0.81       234
           1       0.84      0.99      0.91       390

    accuracy                           0.88       624
   macro avg       0.91      0.84      0.86       624
weighted avg       0.89      0.88      0.87       624



In [58]:
model.save_weights(
    r"Peadiatric_ChesRay_App\mobilenetv2_pneumonia_weights.h5"
)

print("Final MobileNetV2 weights saved")

Final MobileNetV2 weights saved


In [59]:
MODEL_CONFIG = {
    "IMAGE_SIZE": (224,224),
    "THRESHOLD": 0.55,
    "CLASS_NAMES": ['NORMAL', 'PNEUMONIA']
}

print(MODEL_CONFIG)

{'IMAGE_SIZE': (224, 224), 'THRESHOLD': 0.55, 'CLASS_NAMES': ['NORMAL', 'PNEUMONIA']}


In [60]:
import json

with open(
    r"Peadiatric_ChesRay_App\model_config.json",
    "w"
) as f:
    
    json.dump(MODEL_CONFIG, f)

print("Configuration saved successfully")

Configuration saved successfully


#### Verify inside folder

In [61]:
import os

print(os.listdir("Peadiatric_ChesRay_App"))

['mobilenetv2_pneumonia_weights.h5', 'model_config.json']


In [1]:
import os
print(os.getcwd())

C:\Users\HP PC\Data_Science\Medical_Imaging
